In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from tritonoa.data.reader import read_hdf5
from scipy.signal import find_peaks, find_peaks_cwt, hilbert

from vineyard.align_detections import correlate_all_references, merge_correlations, correlations_to_dataframe
from vineyard.config import get_path

In [ ]:
streams = {}
for name in ["3dvha", "vla1", "vla2"]:
    # streams[name], channel = read_hdf5(get_path("denoised_data") / f"{name}.h5"), 1
    streams[name], channel = read_hdf5(get_path("pulse_comp_data") / f"{name}_pc.h5").trim(starttime=np.datetime64("2023-12-01T22:14:00")), 0

In [ ]:
fig, axes = plt.subplots(nrows=3, sharex=True, figsize=(24, 6))

for ax, (name, ds) in zip(axes, streams.items()):
    ax.plot(ds.time_vector, ds.data[channel], label=name)

ax.set_xlabel("Time")
ax.set_ylabel("Amplitude")
plt.tight_layout()
plt.show()

In [ ]:
heights = [0.15, 0.15, 0.15]

fig, axes = plt.subplots(nrows=3, figsize=(24, 6))

times = {}
for ax, height, (name, ds) in zip(axes, heights, streams.items()):
    print(f"Processing {name} with height {height}")
    data = np.abs(hilbert(streams[name].copy().data[channel]))
    peaks, _ = find_peaks(data, distance=3 * ds.stats.sampling_rate, height=height)
    times[name] = ds.time_vector[peaks]
    print(f"Found {len(peaks)} peaks")

    ax.plot(ds.time_vector, data)
    ax.plot(ds.time_vector[peaks], data[peaks], "x")

plt.show()

In [ ]:
plt.close("all")

In [ ]:


t_3dvha_vla1, t_3dvha_vla2, t_vla1_vla2 = get_time_gates(get_path("distance_lut"))
print(f"3DVHA to VLA1 time gate: {t_3dvha_vla1} s")
print(f"3DVHA to VLA2 time gate: {t_3dvha_vla2} s")
print(f"VLA1 to VLA2 time gate: {t_vla1_vla2} s")

time_gates = {
    ("3dvha", "vla1"): t_3dvha_vla1,
    ("3dvha", "vla2"): t_3dvha_vla2,
    ("vla1", "vla2"): t_vla1_vla2,
}

In [ ]:
all_results = correlate_all_references(times, time_gates)
merged_results = merge_correlations(all_results)
merged_results

In [ ]:
df_tdoa = correlations_to_dataframe(merged_results)
df_tdoa